In [2]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

In [3]:
nav = pd.read_csv("../data/raw/02_nav_history.csv")

transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")

performance = pd.read_csv("../data/raw/07_scheme_performance.csv")



In [4]:
nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [6]:
transactions.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [7]:
performance.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [8]:
nav['date'].dtype


<StringDtype(storage='python', na_value=nan)>

In [9]:
nav['date'] = pd.to_datetime(nav['date'])


In [10]:
nav['date'].dtype



dtype('<M8[us]')

In [17]:
nav=nav.sort_values(['amfi_code','date'])

In [18]:
nav = nav.drop_duplicates()

In [19]:
nav.isnull().sum()


amfi_code    0
date         0
nav          0
dtype: int64

In [20]:
nav['nav']=nav.groupby('amfi_code')['nav'].ffill()

In [21]:
invalid_nav = nav[nav['nav']<=0]

invalid_nav

,amfi_code,date,nav


In [23]:
nav.to_csv(
    "../data/processed/nav_history_clean.csv",
    index=False
)

In [24]:
transactions['transaction_date'] = pd.to_datetime(
    transactions['transaction_date']
)

In [25]:
transactions['transaction_type'].unique()


<StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str

In [28]:
transactions['transaction_type']=(
    transactions['transaction_type']
    .str.strip()
    .str.lower()
)

In [29]:
transactions['transaction_type'].unique()


<StringArray>
['sip', 'redemption', 'lumpsum']
Length: 3, dtype: str

In [32]:
transactions['transaction_type'] = (
transactions['transaction_type']
.replace({
'sip':'SIP',
'lumpsum':'Lumpsum',
'redemption':'Redemption'
})
)

In [33]:
transactions['transaction_type'].unique()


<StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str

In [38]:
invalid_transaction =transactions[transactions['amount_inr']<=0]

In [39]:
invalid_transaction

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status


In [40]:
transactions['kyc_status'].unique()

<StringArray>
['Verified', 'Pending']
Length: 2, dtype: str

In [41]:
transactions[
~transactions['kyc_status'].isin(
['Verified','Pending','Rejected']
)
]

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status


In [42]:
transactions.to_csv(
"../data/processed/investor_transactions_clean.csv",
index=False
)

In [43]:
transactions['kyc_status'].unique()

<StringArray>
['Verified', 'Pending']
Length: 2, dtype: str

In [46]:
performance['return_1yr_pct']=pd.to_numeric(
performance['return_1yr_pct'],
errors='coerce'
)
performance['return_3yr_pct']=pd.to_numeric(
performance['return_3yr_pct'],
errors='coerce'
)

performance['return_5yr_pct']=pd.to_numeric(
performance['return_5yr_pct'],
errors='coerce'
)

In [45]:
performance.columns


Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='str')

In [48]:
performance[
performance['return_1yr_pct']>100
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [49]:
performance[
performance['return_3yr_pct']>100
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [50]:
performance[
performance['return_5yr_pct']>100
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [53]:
performance[
(performance['expense_ratio_pct']<0.1)
|
(performance['expense_ratio_pct']>2.5)
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [52]:
performance.columns

Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='str')

In [54]:
performance.to_csv(
"../data/processed/scheme_performance_clean.csv",
index=False
)

In [59]:
from sqlalchemy import create_engine
engine = create_engine("sqlite:///mutual_fund.db")

In [60]:
nav.to_sql(
"nav_history",
engine,
if_exists="replace",
index=False
)

46000

In [61]:
performance.to_sql(
"scheme_performance",
engine,
if_exists="replace",
index=False
)

40

In [62]:
transactions.to_sql(
"investor_transactions",
engine,
if_exists="replace",
index=False
)

32778

In [63]:
print(len(nav))

print(len(transactions))

print(len(performance))

46000
32778
40


In [67]:
import os

print(os.getcwd())

C:\BF\mutual_fund_analytics\notebooks


In [73]:
import sqlite3

conn = sqlite3.connect("../mutual_fund.db")

with open("../sql/schema.sql", "r") as f:
    conn.executescript(f.read())

conn.commit()
conn.close()

print("Schema created successfully!")

Schema created successfully!


In [74]:
import sqlite3

conn = sqlite3.connect("../mutual_fund.db")

cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

tables = cursor.fetchall()

for table in tables:
    print(table[0])

conn.close()

dim_fund
dim_date
fact_nav
fact_transactions
fact_performance
fact_aum
